# Geometric Connections

Connecting prime factorization structure to the geometric properties of Collatz orbits.
We examine Pythagorean triples, complex plane mappings, and proportional power ratios,
looking for systematic differences between primes and composites.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import matplotlib.cm as cm

from collatz.core import orbit, total_stopping_time, stopping_time, stopping_destination
from collatz.dropping import (
    dropping_time, dropping_set, orbital_oddity
)
from collatz.stopping import stopping_class, stopping_point
from collatz.factorization import (
    prime_factorization, is_prime, prime_class_character,
    compare_composite_vs_factors
)
from collatz.geometry import (
    orbital_triple, complex_multiplier, complex_multiplier_exact,
    incircle_params, circumcircle_params, proportional_power_ratio
)

plt.rcParams['figure.figsize'] = (12, 8)
print('All imports successful.')

## Pythagorean Triples: Primes vs Composites

In [ ]:
# Compute orbital triples for integers up to 500.
# Compare distributions of (a, b, c) between primes and composites.

def classify_number(n):
    """Classify n as prime, semiprime, prime power, or other composite."""
    if is_prime(n):
        return 'prime'
    factors = prime_factorization(n)
    if len(factors) == 1:
        # single prime factor => prime power
        return 'prime_power'
    if len(factors) == 2 and all(e == 1 for e in factors.values()):
        return 'semiprime'
    return 'other_composite'

rows = []
for n in range(3, 501, 2):  # odd numbers only (even triples are degenerate)
    a, b, c = orbital_triple(n)
    d = stopping_destination(n)
    category = classify_number(n)
    rows.append({
        'n': n,
        'category': category,
        'a': a, 'b': b, 'c': c,
        'd': d,
        'ratio_a_c': a / c if c != 0 else 0,
        'ratio_b_c': b / c if c != 0 else 0,
        'dropping_set': dropping_set(n),
    })

df_triples = pd.DataFrame(rows)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

categories = ['prime', 'semiprime', 'prime_power', 'other_composite']
colors = {'prime': 'blue', 'semiprime': 'orange', 'prime_power': 'green', 'other_composite': 'red'}

# Plot 1: a vs c by category
for cat in categories:
    subset = df_triples[df_triples['category'] == cat]
    axes[0, 0].scatter(subset['a'], subset['c'], alpha=0.4, s=15,
                       c=colors[cat], label=cat)
axes[0, 0].set_xlabel('a (leg 1)')
axes[0, 0].set_ylabel('c (hypotenuse)')
axes[0, 0].set_title('Orbital Triples: a vs c')
axes[0, 0].legend(fontsize=8)

# Plot 2: b vs c by category
for cat in categories:
    subset = df_triples[df_triples['category'] == cat]
    axes[0, 1].scatter(subset['b'], subset['c'], alpha=0.4, s=15,
                       c=colors[cat], label=cat)
axes[0, 1].set_xlabel('b (leg 2)')
axes[0, 1].set_ylabel('c (hypotenuse)')
axes[0, 1].set_title('Orbital Triples: b vs c')
axes[0, 1].legend(fontsize=8)

# Plot 3: Distribution of a/c ratio by category
for cat in categories:
    subset = df_triples[df_triples['category'] == cat]
    axes[1, 0].hist(subset['ratio_a_c'], bins=30, alpha=0.5,
                    color=colors[cat], label=cat, density=True)
axes[1, 0].set_xlabel('a/c ratio')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Distribution of a/c (leg/hypotenuse)')
axes[1, 0].legend(fontsize=8)

# Plot 4: Distribution of b/c ratio by category
for cat in categories:
    subset = df_triples[df_triples['category'] == cat]
    axes[1, 1].hist(subset['ratio_b_c'], bins=30, alpha=0.5,
                    color=colors[cat], label=cat, density=True)
axes[1, 1].set_xlabel('b/c ratio')
axes[1, 1].set_ylabel('Density')
axes[1, 1].set_title('Distribution of b/c (leg/hypotenuse)')
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Summary statistics
print("\nSummary by category:")
print(df_triples.groupby('category')[['ratio_a_c', 'ratio_b_c']].describe().round(4))

## Complex Plane by Factorization Type

In [ ]:
# Plot z_0 (complex multiplier) on the complex plane,
# grouped by: primes, semiprimes, prime powers, other composites.

rows_z = []
for n in range(3, 501, 2):  # odd numbers
    z0 = complex_multiplier(n)
    category = classify_number(n)
    rows_z.append({
        'n': n,
        'category': category,
        'real': z0.real,
        'imag': z0.imag,
        'dropping_set': dropping_set(n),
    })

df_z = pd.DataFrame(rows_z)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: all points colored by category
markers = {'prime': 'o', 'semiprime': 's', 'prime_power': '^', 'other_composite': 'D'}
for cat in categories:
    subset = df_z[df_z['category'] == cat]
    axes[0].scatter(subset['real'], subset['imag'],
                    alpha=0.5, s=25, c=colors[cat],
                    marker=markers[cat], label=f'{cat} ({len(subset)})')

axes[0].set_xlabel('Re(z_0)')
axes[0].set_ylabel('Im(z_0)')
axes[0].set_title('Complex Multiplier z_0 by Number Type')
axes[0].legend(fontsize=8)
axes[0].axvline(x=0.5, color='gray', linestyle='--', alpha=0.3)
axes[0].set_xlim(0.45, 0.55)

# Right: imaginary part distribution by category
for cat in categories:
    subset = df_z[df_z['category'] == cat]
    axes[1].hist(subset['imag'], bins=40, alpha=0.5,
                 color=colors[cat], label=cat, density=True)

axes[1].set_xlabel('Im(z_0)')
axes[1].set_ylabel('Density')
axes[1].set_title('Distribution of Im(z_0) by Number Type')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# The real part of z_0 is always 1/2; the imaginary part carries the information.
print("\nIm(z_0) statistics by category:")
print(df_z.groupby('category')['imag'].describe().round(4))

## Geometric Clustering

In [ ]:
# Do numbers with similar prime class characters cluster in the complex plane?
# We assign each odd number a "prime class signature" based on its factors' dropping sets,
# then check whether numbers sharing a signature cluster geometrically.

# Build signature: tuple of (dropping_set(p), exponent) for each prime factor, sorted
def factor_class_sig(n):
    """Hashable signature of factor classifications."""
    factors = prime_factorization(n)
    return tuple(sorted((dropping_set(p), exp) for p, exp in factors.items()))

rows_cluster = []
for n in range(3, 301, 2):
    z0 = complex_multiplier(n)
    sig = factor_class_sig(n)
    rows_cluster.append({
        'n': n,
        'real': z0.real,
        'imag': z0.imag,
        'signature': str(sig),
        'category': classify_number(n),
        'dropping_set': dropping_set(n),
    })

df_cluster = pd.DataFrame(rows_cluster)

# Find the most common signatures
sig_counts = df_cluster['signature'].value_counts()
top_sigs = sig_counts.head(8).index.tolist()

fig, ax = plt.subplots(figsize=(12, 8))

cmap = cm.get_cmap('tab10')
for idx, sig in enumerate(top_sigs):
    subset = df_cluster[df_cluster['signature'] == sig]
    ax.scatter(subset['imag'], subset['dropping_set'],
               alpha=0.7, s=40, color=cmap(idx),
               label=f'sig={sig} (n={len(subset)})')

# Also plot "other" signatures in gray
other = df_cluster[~df_cluster['signature'].isin(top_sigs)]
ax.scatter(other['imag'], other['dropping_set'],
           alpha=0.2, s=15, color='gray', label=f'other ({len(other)})')

ax.set_xlabel('Im(z_0)')
ax.set_ylabel('Dropping Set')
ax.set_title('Factor Class Signatures in Im(z_0) vs Dropping Set Space')
ax.legend(fontsize=7, loc='upper left', bbox_to_anchor=(1.01, 1))

plt.tight_layout()
plt.show()

# Visual inspection: do same-signature points form clusters?
print("\nTop factor class signatures and their Im(z_0) ranges:")
for sig in top_sigs:
    subset = df_cluster[df_cluster['signature'] == sig]
    print(f"  {sig}: n={len(subset)}, Im(z_0) in [{subset['imag'].min():.4f}, {subset['imag'].max():.4f}], "
          f"dropping sets: {sorted(subset['dropping_set'].unique())}")

## Proportional Power Ratios

In [ ]:
# Compute P(x) base 2 and base 6 for Collatz orbits.
# Reproduce the polar plot from Paper 3:
# theta = 2*pi*P(x, base_b), r = step index in orbit.

def orbit_ppr_polar(n, base=2):
    """Compute polar coordinates (theta, r) from P(x) along the Collatz orbit."""
    orb = orbit(n)
    thetas = []
    radii = []
    for step, x in enumerate(orb):
        if x >= base:
            p = proportional_power_ratio(x, base)
            thetas.append(2 * np.pi * p)
            radii.append(step)
    return thetas, radii

# Select interesting starting values
test_numbers = [27, 97, 127, 231, 255, 447]

fig, axes = plt.subplots(2, 3, figsize=(16, 10), subplot_kw={'projection': 'polar'})

for idx, n in enumerate(test_numbers):
    row, col = divmod(idx, 3)
    ax = axes[row, col]

    # Base 2
    thetas2, radii2 = orbit_ppr_polar(n, base=2)
    ax.plot(thetas2, radii2, 'b.-', alpha=0.5, markersize=3, linewidth=0.5, label='base 2')

    # Base 6
    thetas6, radii6 = orbit_ppr_polar(n, base=6)
    ax.plot(thetas6, radii6, 'r.-', alpha=0.5, markersize=3, linewidth=0.5, label='base 6')

    cat = classify_number(n)
    ax.set_title(f'n={n} ({cat})\nclass={dropping_set(n)}', fontsize=9)
    ax.legend(fontsize=6, loc='upper right')

plt.suptitle('Polar Plots of P(x) along Collatz Orbits (base 2 vs base 6)', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

# Also: a combined plot with many orbits overlaid
fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={'projection': 'polar'})

# Base 2 combined
for n in range(3, 100, 2):
    thetas, radii = orbit_ppr_polar(n, base=2)
    c = 'blue' if is_prime(n) else 'red'
    axes[0].plot(thetas, radii, '.-', alpha=0.1, markersize=1, linewidth=0.3, color=c)
axes[0].set_title('Base 2: Blue=prime, Red=composite')

# Base 6 combined
for n in range(3, 100, 2):
    thetas, radii = orbit_ppr_polar(n, base=6)
    c = 'blue' if is_prime(n) else 'red'
    axes[1].plot(thetas, radii, '.-', alpha=0.1, markersize=1, linewidth=0.3, color=c)
axes[1].set_title('Base 6: Blue=prime, Red=composite')

plt.suptitle('Overlaid P(x) Polar Plots (odd n from 3 to 99)', y=1.02)
plt.tight_layout()
plt.show()

## Observations

Key findings from this notebook:

1. **Pythagorean Triples**: The orbital triple mapping produces right triangles whose proportions vary with number type. Primes and composites may occupy different regions of the (a/c, b/c) space, reflecting their different dropping destinations relative to their magnitude.

2. **Complex Plane**: The complex multiplier z_0 always has Re(z_0) = 1/2 (a constant). The imaginary part Im(z_0) = (2d - n) / (2n) captures the ratio of the dropping destination to n. Different factorization types (primes, semiprimes, prime powers) may cluster in different ranges of Im(z_0).

3. **Geometric Clustering**: Numbers sharing the same factor class signature (i.e., their prime factors belong to the same Collatz classes) may or may not cluster geometrically. The scatter plot of Im(z_0) vs dropping set reveals whether factorization structure predicts geometric position.

4. **Proportional Power Ratios**: The polar plots of P(x) along orbits visualize how values "rotate" between consecutive powers of the base. Different bases (2 vs 6) produce distinct spiral patterns, and the visual structure may differ between primes and composites.